In [2]:
# 1 - Download, preprocess and split the dataset if necessary. Then create a dataloader for each split

from datasets import load_dataset
import torch
from torch.utils.data import Subset
import random
from IPPy.utilities.metrics import PSNR, SSIM
from modelsnet.network_unet import UNetRes
from image_dataset import ImageDataset
from degradation import ImageDegradation, DegradationParameters, RGBBlurOperator
from utils import plot # type: ignore
from config import TRAIN_SIZE, VALIDATION_SIZE, TEST_SIZE
from IPPy import operators
from models import HQSCG, NAFNet, HQSNet, HQSNetModel
import torch.nn as nn

# Load ImageNet 256x256 from HuggingFace
ds = load_dataset(
    "benjamin-paine/imagenet-1k-256x256"
)

# Apply preprocessing through the custom Dataset class
# Apply preprocessing through the custom Dataset class
train_dataset = Subset(
    ImageDataset(ds["train"]),
    range(TRAIN_SIZE),
)

validation_dataset = Subset(
    ImageDataset(ds["validation"]),
    range(VALIDATION_SIZE),
)

test_dataset = Subset(
    ImageDataset(ds["test"]),
    range(TEST_SIZE),
)

train_degradation_restoration = ImageDegradation(
    DegradationParameters(
        image_size=256,
        kernel_type="motion",
        kernel_size=9,
        motion_angle=45.0,
    )
)

K_rgb = RGBBlurOperator(train_degradation_restoration.operator)

naf_net = NAFNet(
        image_shape=(3,256,256),
        base_channels=32,
        enc_blocks=[1,2,3,4],
        dec_blocks=[2,2,2,1],
        middle_blocks=6
    )
naf_model = NAFNetModel(naf_net)

hqs_net = HQSNet(
    K=K_rgb,
    denoiser=naf_net,
    checkpoint_path="./weights/NAFNet/NAFImgDenoise.pth",
    n_layers=5,
    cg_iters=10,
    initial_mu=1.0,
)

hqs_model = HQSNetModel(hqs_net)

train_degradation_denoiser = ImageDegradation(
    DegradationParameters(
        image_size=256,
        kernel_type=None,
        noise_levels=[0.005, 0.01, 0.05, 0.1]
    )
)

validation_degradations_denoiser = [
    ImageDegradation(DegradationParameters(kernel_type=None, noise_levels=[0.005])),
    ImageDegradation(DegradationParameters(kernel_type=None, noise_levels=[0.01])),
    ImageDegradation(DegradationParameters(kernel_type=None, noise_levels=[0.05])),
    ImageDegradation(DegradationParameters(kernel_type=None, noise_levels=[0.1])),
]

validation_degradations_restoration = [
    ImageDegradation(DegradationParameters(noise_levels=[0.005])),
    ImageDegradation(DegradationParameters(noise_levels=[0.01])),
    ImageDegradation(DegradationParameters(noise_levels=[0.05])),
    ImageDegradation(DegradationParameters(noise_levels=[0.1])),
]

TRAIN_MODEL = False
if TRAIN_MODEL:
    naf_history = naf_model.train_model(
        n_epochs=50,
        train_dataset=train_dataset,
        validation_dataset=validation_dataset,
        train_degradation=train_degradation_denoiser,
        validation_degradations=validation_degradations_denoiser,
        batch_size=16,
        learning_rate=1e-4,
        checkpoint_path="./weights/HQSUnrolled/NAF_checkpoint.pth",
        resume=True,
        preview_every=5,
        preview_n=1
    )


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DRUNET_PATH = "./weights/DRUNet/drunet_color.pth"

drunet = UNetRes(
    in_nc=4,
    out_nc=3,
    nc=[64, 128, 256, 512],
    nb=4,
    act_mode="R",
    downsample_mode="strideconv",
    upsample_mode="convtranspose",
    bias=False
)

state_dict = torch.load(
    DRUNET_PATH,
    map_location=device,
    weights_only=False,
)

drunet.load_state_dict(state_dict, strict=True)
drunet.to(device)
drunet.eval()

for p in drunet.parameters():
    p.requires_grad = False

# 4 - DPIR-style Plug-and-Play HQS with frozen DRUNet denoiser
# Multiple aggressiveness presets.

import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class DRUNetDenoiser(nn.Module):
    def __init__(self, model: nn.Module) -> None:
        super().__init__()
        self.model = model

    def forward(
        self,
        x: torch.Tensor,
        sigma: float | torch.Tensor,
    ) -> torch.Tensor:

        if not torch.is_tensor(sigma):
            sigma = torch.tensor(
                sigma,
                device=x.device,
                dtype=x.dtype,
            )

        sigma_map = torch.ones(
            x.shape[0],
            1,
            x.shape[2],
            x.shape[3],
            device=x.device,
            dtype=x.dtype,
        ) * sigma

        inp = torch.cat([x, sigma_map], dim=1)

        return self.model(inp)


class HQSCG:
    def __init__(self, K) -> None:
        self.K = K

    def batch_dot(
        self,
        a: torch.Tensor,
        b: torch.Tensor,
    ) -> torch.Tensor:
        return torch.sum(a * b, dim=(1, 2, 3), keepdim=True)

    def batch_norm(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        return torch.sqrt(torch.sum(x * x, dim=(1, 2, 3), keepdim=True))

    def apply_normal_operator(
        self,
        x: torch.Tensor,
        rho: torch.Tensor,
    ) -> torch.Tensor:
        return self.K.T(self.K(x)) + rho * x

    def __call__(
        self,
        y_delta: torch.Tensor,
        z: torch.Tensor,
        rho: float | torch.Tensor,
        starting_point: torch.Tensor | None = None,
        maxiter: int = 30,
        tol: float = 1e-5,
        verbose: bool = False,
    ) -> tuple[torch.Tensor, dict[str, list[float]]]:

        y_delta = y_delta.float()
        z = z.float()

        if not torch.is_tensor(rho):
            rho = torch.tensor(
                rho,
                device=y_delta.device,
                dtype=torch.float32,
            )

        rho = rho.float().view(1, 1, 1, 1)

        if starting_point is None:
            x = z.clone()
        else:
            x = starting_point.float().clone()

        b = self.K.T(y_delta) + rho * z

        r = b - self.apply_normal_operator(x, rho)
        p = r.clone()

        rs_old = self.batch_dot(r, r)
        rs_initial = rs_old.clone()

        info: dict[str, list[float]] = {
            "relative_system_residual": [],
            "data_residual": [],
        }

        eps = 1e-8

        for k in range(maxiter):
            relative_system_residual = torch.sqrt(
                rs_old / (rs_initial + eps)
            ).mean()

            data_residual = self.batch_norm(
                self.K(x) - y_delta
            ).mean()

            info["relative_system_residual"].append(
                relative_system_residual.item()
            )
            info["data_residual"].append(data_residual.item())

            if verbose:
                print(
                    f"CG iter {k:03d} | "
                    f"rel_sys_res={relative_system_residual.item():.6f} | "
                    f"data_res={data_residual.item():.4f}"
                )

            if relative_system_residual.item() < tol:
                break

            Ap = self.apply_normal_operator(p, rho)
            alpha = rs_old / (self.batch_dot(p, Ap) + eps)

            x = x + alpha * p
            r = r - alpha * Ap

            rs_new = self.batch_dot(r, r)
            beta = rs_new / (rs_old + eps)

            p = r + beta * p
            rs_old = rs_new

        return x, info


def get_dpir_rho_sigma(
    noise_level: float,
    iter_num: int,
    model_sigma_1: float,
    model_sigma_2: float,
    w: float = 1.0,
    rho_scale: float = 0.23,
) -> tuple[list[float], list[float]]:

    sigma_log = np.logspace(
        np.log10(model_sigma_1),
        np.log10(model_sigma_2),
        iter_num,
    ).astype(np.float32)

    sigma_lin = np.linspace(
        model_sigma_1,
        model_sigma_2,
        iter_num,
    ).astype(np.float32)

    sigmas = (sigma_log * w + sigma_lin * (1.0 - w)) / 255.0

    rhos = [
        rho_scale * (noise_level ** 2) / (sigma ** 2)
        for sigma in sigmas
    ]

    return rhos, sigmas.tolist()


drunet_denoiser = DRUNetDenoiser(drunet).to(device)
drunet_denoiser.eval()

for p in drunet_denoiser.parameters():
    p.requires_grad = False


def pnp_hqs_dpir_style(
    y_delta: torch.Tensor,
    K,
    denoiser: DRUNetDenoiser,
    noise_level: float,
    iter_num: int,
    model_sigma_1: float,
    model_sigma_2: float,
    w: float = 1.0,
    rho_scale: float = 0.23,
    cg_iters: int = 30,
    cg_tol: float = 1e-5,
    init_mode: str = "degraded",
) -> tuple[torch.Tensor, dict[str, list]]:

    rhos, sigmas = get_dpir_rho_sigma(
        noise_level=noise_level,
        iter_num=iter_num,
        model_sigma_1=model_sigma_1,
        model_sigma_2=model_sigma_2,
        w=w,
        rho_scale=rho_scale,
    )

    cg_solver = HQSCG(K)

    if init_mode == "degraded":
        x = y_delta.clone().clamp(0.0, 1.0)
    elif init_mode == "adjoint":
        x = K.T(y_delta).clamp(0.0, 1.0)
    else:
        raise ValueError("init_mode must be either 'degraded' or 'adjoint'.")

    intermediates: dict[str, list] = {
        "x": [x.detach().cpu()],
        "x_data": [],
        "z": [],
        "rho": [],
        "sigma": [],
        "cg_relative_system_residuals": [],
        "cg_data_residuals": [],
    }

    for rho_value, sigma_value in zip(rhos, sigmas):

        # DPIR step 1: data fidelity
        with torch.no_grad():
            with torch.amp.autocast("cuda", enabled=False):  # type: ignore
                x_data, cg_info = cg_solver(
                    y_delta=y_delta.float(),
                    z=x.float(),
                    rho=rho_value,
                    starting_point=x.float(),
                    maxiter=cg_iters,
                    tol=cg_tol,
                    verbose=False,
                )

        x_data = x_data.clamp(0.0, 1.0)

        # DPIR step 2: denoising prior
        with torch.no_grad():
            x = denoiser(
                x_data,
                sigma=sigma_value,
            ).clamp(0.0, 1.0)

        intermediates["x_data"].append(x_data.detach().cpu())
        intermediates["z"].append(x.detach().cpu())
        intermediates["x"].append(x.detach().cpu())
        intermediates["rho"].append(torch.tensor(rho_value).detach().cpu())
        intermediates["sigma"].append(torch.tensor(sigma_value).detach().cpu())
        intermediates["cg_relative_system_residuals"].append(
            cg_info["relative_system_residual"]
        )
        intermediates["cg_data_residuals"].append(
            cg_info["data_residual"]
        )

    return x, intermediates


def print_metrics(
    name: str,
    pred: torch.Tensor,
    clean: torch.Tensor,
) -> tuple[float, float]:

    psnr_value = PSNR(pred.float(), clean.float())
    ssim_value = SSIM(pred.float(), clean.float())

    if isinstance(psnr_value, torch.Tensor):
        psnr_value = psnr_value.item()

    if isinstance(ssim_value, torch.Tensor):
        ssim_value = ssim_value.item()

    print(f"{name}: PSNR={psnr_value:.4f} | SSIM={ssim_value:.4f}")

    return psnr_value, ssim_value


# ------------------------------------------------------------
# Test image and degradation
# ------------------------------------------------------------

clean = test_dataset[1]

if isinstance(clean, tuple):
    clean = clean[0]

clean = clean.unsqueeze(0).to(device)

noise_level = 0.05

test_degradation = ImageDegradation(
    DegradationParameters(
        image_size=256,
        kernel_type="motion",
        kernel_size=9,
        motion_angle=45.0,
        noise_levels=[noise_level],
    )
)

with torch.no_grad():
    degraded = test_degradation(clean).to(device)


# ------------------------------------------------------------
# Presets
# ------------------------------------------------------------

presets = {
    # Meno aggressivo: meno sigma alta iniziale, meno iterazioni.
    "soft_25_15": {
        "iter_num": 5,
        "model_sigma_1": 25.0,
        "model_sigma_2": 15.0,
        "w": 1.0,
        "rho_scale": 0.23,
        "cg_iters": 30,
    },

    # Compromesso probabile.
    "medium_35_15": {
        "iter_num": 6,
        "model_sigma_1": 35.0,
        "model_sigma_2": 15.0,
        "w": 1.0,
        "rho_scale": 0.23,
        "cg_iters": 30,
    },

    # DPIR-like originale, più aggressivo.
    "strong_49_12_75": {
        "iter_num": 8,
        "model_sigma_1": 49.0,
        "model_sigma_2": noise_level * 255.0,
        "w": 1.0,
        "rho_scale": 0.23,
        "cg_iters": 30,
    },

    # Meno prior / più data fidelity rispetto a DPIR.
    "medium_low_rho": {
        "iter_num": 6,
        "model_sigma_1": 35.0,
        "model_sigma_2": 15.0,
        "w": 1.0,
        "rho_scale": 0.12,
        "cg_iters": 30,
    },

    # Più prior / più smoothing.
    "medium_high_rho": {
        "iter_num": 6,
        "model_sigma_1": 35.0,
        "model_sigma_2": 15.0,
        "w": 1.0,
        "rho_scale": 0.35,
        "cg_iters": 30,
    },
}


# ------------------------------------------------------------
# Run all presets
# ------------------------------------------------------------

results = {}

print("----- Baseline -----")
print_metrics("Degraded", degraded, clean)

for name, cfg in presets.items():
    print(f"\n----- Running preset: {name} -----")

    with torch.no_grad():
        restored, intermediates = pnp_hqs_dpir_style(
            y_delta=degraded,
            K=K_rgb,
            denoiser=drunet_denoiser,
            noise_level=noise_level,
            iter_num=cfg["iter_num"],
            model_sigma_1=cfg["model_sigma_1"],
            model_sigma_2=cfg["model_sigma_2"],
            w=cfg["w"],
            rho_scale=cfg["rho_scale"],
            cg_iters=cfg["cg_iters"],
            cg_tol=1e-5,
            init_mode="degraded",
        )

    psnr_value, ssim_value = print_metrics(name, restored, clean)

    results[name] = {
        "restored": restored,
        "intermediates": intermediates,
        "psnr": psnr_value,
        "ssim": ssim_value,
    }

    print("Schedule:")
    for i in range(len(intermediates["rho"])):
        rho = intermediates["rho"][i].item()
        sigma = intermediates["sigma"][i].item()
        rel_res = intermediates["cg_relative_system_residuals"][i]
        data_res = intermediates["cg_data_residuals"][i]

        print(
            f"  iter {i + 1} | "
            f"rho={rho:.6f} | "
            f"sigma={sigma:.6f} | "
            f"CG={len(rel_res)} | "
            f"rel_sys_res={rel_res[-1]:.6f} | "
            f"data_res={data_res[-1]:.4f}"
        )


# ------------------------------------------------------------
# Final comparison plot
# ------------------------------------------------------------

images = [
    clean[0].detach().cpu(),
    degraded[0].detach().cpu(),
]

titles = [
    "Clean",
    "Degraded",
]

for name, result in results.items():
    images.append(result["restored"][0].detach().cpu())
    titles.append(
        f"{name}\n"
        f"PSNR={result['psnr']:.2f}\n"
        f"SSIM={result['ssim']:.3f}"
    )

plot(
    *images,
    titles=titles,
)


# ------------------------------------------------------------
# Iteration plots for each preset
# ------------------------------------------------------------

for name, result in results.items():
    intermediates = result["intermediates"]

    images = [
        clean[0].detach().cpu(),
        degraded[0].detach().cpu(),
    ]

    titles = [
        "Clean",
        "Degraded",
    ]

    for i, x_iter in enumerate(intermediates["x"][1:]):
        rho_value = intermediates["rho"][i].item()
        sigma_value = intermediates["sigma"][i].item()

        images.append(x_iter[0])
        titles.append(
            f"{name}\n"
            f"iter {i + 1}\n"
            f"rho={rho_value:.3f}\n"
            f"sigma={sigma_value:.3f}"
        )

    plot(
        *images,
        titles=titles,
    )

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/36 [00:00<?, ?it/s]

NameError: name 'NAFNetModel' is not defined